In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

from src.load import loadMatches, loadEvents, LEAKS

In [2]:
matches = loadMatches()
wc = matches[(matches["competition"] == "International - FIFA World Cup") & (matches["season"] == "2022")]
print(len(wc), "matches")

raw = []
for m in wc["match_id"]:
    ev = loadEvents(m)
    raw.append(ev[ev["type"] == "Shot"])
raw = pd.concat(raw, ignore_index=True)
print(raw.shape[0], "shots,", raw.shape[1], "columns")

64 matches
1494 shots, 114 columns


In [3]:
print("rows left after dropna():", len(raw.dropna()))

# flags only get written when they happened, so a blank means no, not missing
flags = ["under_pressure", "shot_first_time", "shot_one_on_one", "shot_open_goal",
         "shot_aerial_won", "shot_follows_dribble", "shot_redirect", "shot_deflected"]
for f in flags:
    if f in raw.columns:
        print(f, "blank on", round(raw[f].isna().mean() * 100, 1), "% of shots")

print(raw["shot_type"].value_counts())
print(raw["period"].value_counts().sort_index())  # 5 is the shootout

rows left after dropna(): 0
under_pressure blank on 84.1 % of shots
shot_first_time blank on 68.7 % of shots
shot_one_on_one blank on 94.3 % of shots
shot_open_goal blank on 99.1 % of shots
shot_aerial_won blank on 91.4 % of shots
shot_follows_dribble blank on 99.8 % of shots
shot_redirect blank on 99.9 % of shots
shot_deflected blank on 98.5 % of shots
shot_type
Open Play    1382
Penalty        64
Free Kick      46
Corner          2
Name: count, dtype: int64
period
1    607
2    801
3     20
4     25
5     41
Name: count, dtype: int64


In [4]:
# the naive way: every column, text turned into dummies, blanks filled with 0, random split by shot
y = (raw["shot_outcome"] == "Goal").astype(int)

keep = []
for c in raw.columns:
    vals = raw[c].dropna()
    if len(vals) == 0:
        continue
    # lists and dicts (locations, freeze frames) can't go in as they are
    if isinstance(vals.iloc[0], (list, dict, np.ndarray)):
        continue
    # ids, timestamps and names have too many values to be dummies
    if not pd.api.types.is_numeric_dtype(raw[c]) and raw[c].nunique() > 50:
        continue
    keep.append(c)

X = pd.get_dummies(raw[keep]).fillna(0).astype(float)
print(X.shape[1], "columns after dummies")

Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=2026)
naive = XGBClassifier(n_estimators=200, max_depth=3).fit(Xa, ya)
print("naive AUC:", round(roc_auc_score(yb, naive.predict_proba(Xb)[:, 1]), 3))

141 columns after dummies
naive AUC: 1.0


In [5]:
imp = pd.Series(naive.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(8).round(3)

shot_outcome_Goal     0.997
shot_statsbomb_xg     0.003
index                 0.000
minute                0.000
match_id              0.000
period                0.000
player_id             0.000
possession_team_id    0.000
dtype: float32

In [6]:
# drop every column that comes from a leak
leakCols = []
for c in X.columns:
    for leak in LEAKS:
        if c == leak or c.startswith(leak + "_"):
            leakCols.append(c)
print("leak columns:", leakCols)

X2 = X.drop(columns=leakCols)
Xa, Xb, ya, yb = train_test_split(X2, y, test_size=0.25, random_state=2026)
honest = XGBClassifier(n_estimators=200, max_depth=3).fit(Xa, ya)
print("without leaks:", round(roc_auc_score(yb, honest.predict_proba(Xb)[:, 1]), 3))

leak columns: ['shot_statsbomb_xg', 'out_True', 'shot_outcome_Blocked', 'shot_outcome_Goal', 'shot_outcome_Off T', 'shot_outcome_Post', 'shot_outcome_Saved', 'shot_outcome_Saved Off Target', 'shot_outcome_Saved to Post', 'shot_outcome_Wayward', 'shot_deflected_True', 'shot_saved_to_post_True', 'shot_saved_off_target_True']
without leaks: 0.7
